In [1]:
# %% [markdown]
# # 00 — Data Preparation · Clasificación
# **Objetivo:** Limpiar y unir los datasets de Intakes y Outcomes,
# crear la variable objetivo con 4 clases y generar las features necesarias.
#
# **Output:** `data/animal_classification.csv`

# %% [markdown]
# ## Imports

# %%
import polars as pl
import polars.selectors as cs
from datetime import datetime

In [2]:
# %% [markdown]
# ---
# ## 1. Carga de datos

# %%
intakes  = pl.read_csv("../data/raw/Austin_Animal_Center_Intakes.csv",
                       try_parse_dates=False)
outcomes = pl.read_csv("../data/raw/Austin_Animal_Center_Outcomes.csv",
                       try_parse_dates=False)

print(f"Intakes:  {intakes.shape}")
print(f"Outcomes: {outcomes.shape}")
print("\nColumnas Intakes:")
print(intakes.columns)
print("\nColumnas Outcomes:")
print(outcomes.columns)

Intakes:  (124120, 12)
Outcomes: (124491, 12)

Columnas Intakes:
['Animal ID', 'Name', 'DateTime', 'MonthYear', 'Found Location', 'Intake Type', 'Intake Condition', 'Animal Type', 'Sex upon Intake', 'Age upon Intake', 'Breed', 'Color']

Columnas Outcomes:
['Animal ID', 'Name', 'DateTime', 'MonthYear', 'Date of Birth', 'Outcome Type', 'Outcome Subtype', 'Animal Type', 'Sex upon Outcome', 'Age upon Outcome', 'Breed', 'Color']


In [3]:
# %% [markdown]
# ---
# ## 2. Parsear fechas
#
# Necesitamos las fechas como tipo datetime para:
# - Hacer el join cronológico (cada ingreso → su siguiente outcome)
# - Calcular DaysInShelter (solo EDA)
# - Extraer el mes para crear la feature Season

# %%
FMT = "%m/%d/%Y %I:%M:%S %p"

intakes = intakes.with_columns(
    pl.col("DateTime")
      .str.to_datetime(FMT, strict=False)
      .alias("DateTime")
)

outcomes = outcomes.with_columns(
    pl.col("DateTime")
      .str.to_datetime(FMT, strict=False)
      .alias("DateTime")
)

print("✅ Fechas parseadas correctamente")
print(f"   Intakes  DateTime nulls: {intakes['DateTime'].null_count()}")
print(f"   Outcomes DateTime nulls: {outcomes['DateTime'].null_count()}")

✅ Fechas parseadas correctamente
   Intakes  DateTime nulls: 0
   Outcomes DateTime nulls: 0


In [ ]:

# ---
# ## 3. Join inteligente
#
# **Problema:** un animal puede entrar y salir del refugio varias veces.
# Si hacemos un join directo por Animal ID obtenemos filas duplicadas
# (cada ingreso cruzado con cada outcome).
#
# **Solución:** para cada ingreso, buscamos el outcome más cercano
# en el tiempo que ocurra DESPUÉS de la fecha de ingreso.
# Así emparejamos correctamente cada entrada con su salida.

# %%
merged = (
    intakes
    .join(outcomes, on="Animal ID", suffix="_outcome")
    .filter(pl.col("DateTime_outcome") >= pl.col("DateTime"))
    .sort("DateTime_outcome")
    .unique(subset=["Animal ID", "DateTime"], keep="first")
)

print(f"Shape tras join inteligente: {merged.shape}")
print("\nDistribución Outcome Type antes de agrupar:")
print(merged["Outcome Type"].value_counts().sort("count", descending=True))

Shape tras join inteligente: (123001, 23)

Distribución Outcome Type antes de agrupar:
shape: (10, 2)
┌─────────────────┬───────┐
│ Outcome Type    ┆ count │
│ ---             ┆ ---   │
│ str             ┆ u32   │
╞═════════════════╪═══════╡
│ Adoption        ┆ 54714 │
│ Transfer        ┆ 36125 │
│ Return to Owner ┆ 21413 │
│ Euthanasia      ┆ 8269  │
│ Died            ┆ 1121  │
│ Rto-Adopt       ┆ 699   │
│ Disposal        ┆ 556   │
│ Missing         ┆ 67    │
│ Relocate        ┆ 19    │
│ null            ┆ 18    │
└─────────────────┴───────┘


In [5]:
# %% [markdown]
# ---
# ## 4. Filtrar tipo de animal
#
# Nos quedamos solo con Dog y Cat.
# Other, Bird y Livestock representan menos del 6% y tienen
# distribuciones de outcome muy distintas que añadirían ruido.

# %%
merged = merged.filter(
    pl.col("Animal Type").is_in(["Dog", "Cat"])
)

print(f"Shape tras filtrar Animal Type: {merged.shape}")
print(merged["Animal Type"].value_counts())

Shape tras filtrar Animal Type: (115855, 23)
shape: (2, 2)
┌─────────────┬───────┐
│ Animal Type ┆ count │
│ ---         ┆ ---   │
│ str         ┆ u32   │
╞═════════════╪═══════╡
│ Cat         ┆ 45986 │
│ Dog         ┆ 69869 │
└─────────────┴───────┘


In [6]:
# %% [markdown]
# ---
# ## 5. Variable objetivo — 4 clases
#
# Agrupamos los 9 outcomes originales en 4 clases con sentido de negocio:
#
# | Clase           | Incluye                          | Justificación                          |
# |-----------------|----------------------------------|----------------------------------------|
# | Adoption        | Adoption + Rto-Adopt             | Ambas son adopciones                   |
# | Transfer        | Transfer                         | Traslado a otro centro                 |
# | Return to Owner | Return to Owner                  | Devolución al dueño                    |
# | At Risk         | Euthanasia + Died + Disposal     | Cualquier resultado negativo           |
#
# Missing y Relocate (< 90 registros) se eliminan por ser ruido.

# %%
merged = merged.with_columns(
    pl.when(pl.col("Outcome Type").is_in(["Adoption", "Rto-Adopt"]))
        .then(pl.lit("Adoption"))
    .when(pl.col("Outcome Type") == "Transfer")
        .then(pl.lit("Transfer"))
    .when(pl.col("Outcome Type") == "Return to Owner")
        .then(pl.lit("Return to Owner"))
    .when(pl.col("Outcome Type").is_in(["Euthanasia", "Died", "Disposal"]))
        .then(pl.lit("At Risk"))
    .otherwise(None)
    .alias("OutcomeClass")
)

merged = merged.filter(pl.col("OutcomeClass").is_not_null())

print("Distribución final de clases:")
counts = merged["OutcomeClass"].value_counts().sort("count", descending=True)
print(counts)
total = len(merged)
for row in counts.iter_rows():
    clase, n = row
    print(f"  {clase:<20} {n:>6,}  ({n/total*100:.1f}%)")

Distribución final de clases:
shape: (4, 2)
┌─────────────────┬───────┐
│ OutcomeClass    ┆ count │
│ ---             ┆ ---   │
│ str             ┆ u32   │
╞═════════════════╪═══════╡
│ Adoption        ┆ 54679 │
│ Transfer        ┆ 35084 │
│ Return to Owner ┆ 21330 │
│ At Risk         ┆ 4684  │
└─────────────────┴───────┘
  Adoption             54,679  (47.2%)
  Transfer             35,084  (30.3%)
  Return to Owner      21,330  (18.4%)
  At Risk               4,684  (4.0%)


In [7]:
# %% [markdown]
# ---
# ## 6. Feature — DaysInShelter
#
# ⚠️ SOLO PARA EDA — NO se usará como feature del modelo.
#
# Por qué es una trampa (data leakage):
# Esta variable se calcula como (fecha de salida - fecha de entrada).
# Para calcularla necesitas saber cuándo salió el animal, pero eso
# es parte del outcome que intentamos predecir. En producción,
# cuando llega un animal nuevo, DaysInShelter = 0 porque acaba
# de llegar — el modelo habría aprendido con información que no
# existe en el momento real de la predicción.
#
# Uso válido: análisis exploratorio para entender patrones históricos.

# %%
merged = merged.with_columns(
    (
        (pl.col("DateTime_outcome") - pl.col("DateTime"))
        .dt.total_days()
    ).alias("DaysInShelter")
)

# Sanear valores negativos (errores de registro en los datos originales)
merged = merged.filter(pl.col("DaysInShelter") >= 0)

print("DaysInShelter (solo EDA):")
print(merged["DaysInShelter"].describe())
print(f"\nAnimales con DaysInShelter = 0: {(merged['DaysInShelter'] == 0).sum():,}")
print(f"Animales con DaysInShelter > 90: {(merged['DaysInShelter'] > 90).sum():,}")

DaysInShelter (solo EDA):
shape: (9, 2)
┌────────────┬───────────┐
│ statistic  ┆ value     │
│ ---        ┆ ---       │
│ str        ┆ f64       │
╞════════════╪═══════════╡
│ count      ┆ 115777.0  │
│ null_count ┆ 0.0       │
│ mean       ┆ 18.948055 │
│ std        ┆ 44.128166 │
│ min        ┆ 0.0       │
│ 25%        ┆ 2.0       │
│ 50%        ┆ 5.0       │
│ 75%        ┆ 16.0      │
│ max        ┆ 1521.0    │
└────────────┴───────────┘

Animales con DaysInShelter = 0: 20,766
Animales con DaysInShelter > 90: 4,599


In [8]:
# %% [markdown]
# ---
# ## 7. Feature — Season (estación del año)
#
# Extraemos el mes del DateTime de ingreso y lo mapeamos a
# estaciones del hemisferio norte (referencia: Madrid, España),
# adaptando el proyecto a contexto europeo aunque los datos sean de Texas.
#
# | Estación   | Meses                    |
# |------------|--------------------------|
# | Primavera  | Marzo, Abril, Mayo       |
# | Verano     | Junio, Julio, Agosto     |
# | Otoño      | Septiembre, Octubre, Nov |
# | Invierno   | Diciembre, Enero, Feb    |
#
# Esta feature SÍ entra al modelo — el mes de ingreso es información
# disponible en el momento en que llega el animal.

# %%
merged = merged.with_columns(
    pl.col("DateTime").dt.month().alias("IntakeMonth")
)

merged = merged.with_columns(
    pl.when(pl.col("IntakeMonth").is_in([3, 4, 5]))
        .then(pl.lit("Primavera"))
    .when(pl.col("IntakeMonth").is_in([6, 7, 8]))
        .then(pl.lit("Verano"))
    .when(pl.col("IntakeMonth").is_in([9, 10, 11]))
        .then(pl.lit("Otoño"))
    .otherwise(pl.lit("Invierno"))
    .alias("Season")
)

print("Distribución por estación:")
print(merged["Season"].value_counts().sort("count", descending=True))

Distribución por estación:
shape: (4, 2)
┌───────────┬───────┐
│ Season    ┆ count │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ Verano    ┆ 31667 │
│ Otoño     ┆ 30812 │
│ Primavera ┆ 28463 │
│ Invierno  ┆ 24835 │
└───────────┴───────┘


In [9]:
# %% [markdown]
# ---
# ## 8. Parsear edad a días

# %%
def parse_age(age_str: str) -> float:
    """Convierte strings de edad ('2 years', '3 months'...) a días."""
    if age_str is None:
        return None
    age_str = str(age_str).lower().strip()
    parts = age_str.split()
    if len(parts) < 2:
        return None
    try:
        n = float(parts[0])
    except ValueError:
        return None
    if "year"  in parts[1]: return n * 365
    if "month" in parts[1]: return n * 30
    if "week"  in parts[1]: return n * 7
    if "day"   in parts[1]: return n
    return None

merged = merged.with_columns(
    pl.col("Age upon Intake")
      .map_elements(parse_age, return_dtype=pl.Float64)
      .alias("AgeInDays")
)

# Eliminar edades inválidas
merged = merged.filter(pl.col("AgeInDays") > 0)

print("AgeInDays:")
print(merged["AgeInDays"].describe())

AgeInDays:
shape: (9, 2)
┌────────────┬─────────────┐
│ statistic  ┆ value       │
│ ---        ┆ ---         │
│ str        ┆ f64         │
╞════════════╪═════════════╡
│ count      ┆ 115080.0    │
│ null_count ┆ 0.0         │
│ mean       ┆ 773.687626  │
│ std        ┆ 1074.857786 │
│ min        ┆ 1.0         │
│ 25%        ┆ 60.0        │
│ 50%        ┆ 365.0       │
│ 75%        ┆ 1095.0      │
│ max        ┆ 8760.0      │
└────────────┴─────────────┘


In [10]:
# %% [markdown]
# ---
# ## 9. Features adicionales

# %%
# AgeGroup — grupos de edad con sentido de negocio
merged = merged.with_columns(
    pl.when(pl.col("AgeInDays") < 180)
        .then(pl.lit("Cachorro (<6m)"))
    .when(pl.col("AgeInDays") < 365)
        .then(pl.lit("Joven (6m-1a)"))
    .when(pl.col("AgeInDays") < 1095)
        .then(pl.lit("Adulto joven (1-3a)"))
    .when(pl.col("AgeInDays") < 2555)
        .then(pl.lit("Adulto (3-7a)"))
    .otherwise(pl.lit("Senior (>7a)"))
    .alias("AgeGroup")
)

# breed_type — mestizo o raza pura
merged = merged.with_columns(
    pl.when(pl.col("Breed").str.contains("Mix"))
        .then(pl.lit("mix"))
    .otherwise(pl.lit("purebred"))
    .alias("breed_type")
)

# Color_grouped — número de colores
merged = merged.with_columns(
    pl.when(pl.col("Color").str.contains("/"))
        .then(
            pl.when(pl.col("Color").str.count_matches("/") >= 2)
                .then(pl.lit("Tricolor"))
            .otherwise(pl.lit("Bicolor"))
        )
    .otherwise(pl.lit("Monocolor"))
    .alias("Color_grouped")
)

# Eliminar Sex desconocido
merged = merged.filter(
    ~pl.col("Sex upon Intake").is_in(["Unknown", None])
)

print("✅ Features adicionales creadas")
print(f"   AgeGroup:     {merged['AgeGroup'].n_unique()} categorías")
print(f"   breed_type:   {merged['breed_type'].n_unique()} categorías")
print(f"   Color_grouped:{merged['Color_grouped'].n_unique()} categorías")
print(f"   Season:       {merged['Season'].n_unique()} categorías")

✅ Features adicionales creadas
   AgeGroup:     5 categorías
   breed_type:   2 categorías
   Color_grouped:2 categorías
   Season:       4 categorías


In [11]:
# %% [markdown]
# ---
# ## 10. Seleccionar columnas finales y guardar
#
# Guardamos dos CSVs separados con propósitos distintos:
#
# | Archivo | Contiene DaysInShelter | Uso |
# |---|---|---|
# | animal_classification.csv | ❌ | Entrenar el modelo |
# | animal_eda.csv | ✅ | Análisis exploratorio |
#
# Esta separación física evita que alguien use por error
# DaysInShelter como feature del modelo (data leakage).

# %%
df_model = merged.select([
    pl.col("Animal ID").alias("AnimalID"),
    pl.col("Animal Type").alias("AnimalType"),
    pl.col("Breed").alias("Breed"),
    pl.col("Color").alias("Color"),
    pl.col("Sex upon Intake").alias("Sex"),
    pl.col("Intake Type").alias("IntakeType"),        # ← con espacio en el original
    pl.col("Intake Condition").alias("IntakeCondition"),  # ← con espacio en el original
    pl.col("AgeInDays"),
    pl.col("AgeGroup"),
    pl.col("breed_type"),
    pl.col("Color_grouped"),
    pl.col("Season"),
    pl.col("OutcomeClass"),
])

df_eda = merged.select([
    pl.col("Animal ID").alias("AnimalID"),
    pl.col("Animal Type").alias("AnimalType"),
    pl.col("Breed").alias("Breed"),
    pl.col("Color").alias("Color"),
    pl.col("Sex upon Intake").alias("Sex"),
    pl.col("Intake Type").alias("IntakeType"),
    pl.col("Intake Condition").alias("IntakeCondition"),
    pl.col("AgeInDays"),
    pl.col("AgeGroup"),
    pl.col("breed_type"),
    pl.col("Color_grouped"),
    pl.col("Season"),
    pl.col("IntakeMonth"),
    pl.col("DaysInShelter"),    # ← solo aquí
    pl.col("OutcomeClass"),
])

print(f"Dataset modelo: {df_model.shape}")
print(f"Dataset EDA:    {df_eda.shape}")
print(f"\nNulos en dataset modelo:")
print(df_model.null_count())

# %%
df_model.write_csv("../data/processed/animal_classification.csv")
df_eda.write_csv("../data/processed/animal_eda.csv")

print("\n✅ Datasets guardados:")
print("   → data/processed/animal_classification.csv  (para el modelo)")
print("   → data/processed/animal_eda.csv             (para el EDA, incluye DaysInShelter)")


Dataset modelo: (111055, 13)
Dataset EDA:    (111055, 15)

Nulos en dataset modelo:
shape: (1, 13)
┌──────────┬────────────┬───────┬───────┬───┬────────────┬───────────────┬────────┬──────────────┐
│ AnimalID ┆ AnimalType ┆ Breed ┆ Color ┆ … ┆ breed_type ┆ Color_grouped ┆ Season ┆ OutcomeClass │
│ ---      ┆ ---        ┆ ---   ┆ ---   ┆   ┆ ---        ┆ ---           ┆ ---    ┆ ---          │
│ u32      ┆ u32        ┆ u32   ┆ u32   ┆   ┆ u32        ┆ u32           ┆ u32    ┆ u32          │
╞══════════╪════════════╪═══════╪═══════╪═══╪════════════╪═══════════════╪════════╪══════════════╡
│ 0        ┆ 0          ┆ 0     ┆ 0     ┆ … ┆ 0          ┆ 0             ┆ 0      ┆ 0            │
└──────────┴────────────┴───────┴───────┴───┴────────────┴───────────────┴────────┴──────────────┘

✅ Datasets guardados:
   → data/processed/animal_classification.csv  (para el modelo)
   → data/processed/animal_eda.csv             (para el EDA, incluye DaysInShelter)


In [12]:
# %% [markdown]
# ---
# ## 11. Resumen final

# %%
print("=" * 60)
print("RESUMEN DEL PREPROCESAMIENTO")
print("=" * 60)
print(f"\nRegistros finales:   {len(df_model):,}")
print(f"Features del modelo: {df_model.width - 1}")
print(f"Nulos:               {df_model.null_count().sum_horizontal().sum()}")

print("\nCOLUMNAS DEL MODELO:")
for col in df_model.columns:
    dtype  = str(df_model[col].dtype)
    unique = df_model[col].n_unique()
    marker = " ← TARGET"   if col == "OutcomeClass"   else ""
    print(f"  {col:<22} {dtype:<12} ({unique:>5} únicos){marker}")

print("\nDISTRIBUCIÓN DE CLASES:")
counts = df_model["OutcomeClass"].value_counts().sort("count", descending=True)
total  = len(df_model)
for row in counts.iter_rows():
    clase, n = row
    pct = n / total * 100
    bar = "█" * int(pct / 2)
    print(f"  {clase:<22} {n:>6,}  ({pct:4.1f}%)  {bar}")

print("\nDISTRIBUCIÓN POR ESTACIÓN:")
season = df_model["Season"].value_counts().sort("count", descending=True)
for row in season.iter_rows():
    est, n = row
    pct = n / total * 100
    print(f"  {est:<12} {n:>6,}  ({pct:4.1f}%)")

print("\n⚠️  RECORDATORIO DATA LEAKAGE:")
print("   DaysInShelter → solo en animal_eda.csv, nunca en el modelo")

RESUMEN DEL PREPROCESAMIENTO

Registros finales:   111,055
Features del modelo: 12
Nulos:               0

COLUMNAS DEL MODELO:
  AnimalID               String       (98120 únicos)
  AnimalType             String       (    2 únicos)
  Breed                  String       ( 2415 únicos)
  Color                  String       (  550 únicos)
  Sex                    String       (    5 únicos)
  IntakeType             String       (    5 únicos)
  IntakeCondition        String       (   10 únicos)
  AgeInDays              Float64      (   45 únicos)
  AgeGroup               String       (    5 únicos)
  breed_type             String       (    2 únicos)
  Color_grouped          String       (    2 únicos)
  Season                 String       (    4 únicos)
  OutcomeClass           String       (    4 únicos) ← TARGET

DISTRIBUCIÓN DE CLASES:
  Adoption               54,387  (49.0%)  ████████████████████████
  Transfer               31,351  (28.2%)  ██████████████
  Return to Owner        

In [13]:
# ============================================
# ANÁLISIS DE DUPLICADOS
# ============================================

# Duplicados de fila completa (todas las columnas iguales)
dup_total = merged.filter(merged.is_duplicated())
print(f"Filas duplicadas exactas: {len(dup_total):,}")

# Duplicados por Animal ID (mismo animal, distintas entradas)
dup_id = merged.filter(
    pl.col("Animal ID").is_duplicated()
)
print(f"Animal IDs duplicados: {dup_id['Animal ID'].n_unique():,}")

# Duplicados por Animal ID + fecha de ingreso exacta
# (esto sí es un error real — mismo animal, mismo día)
dup_exact = merged.filter(
    pl.struct(["Animal ID", "DateTime"]).is_duplicated()
)
print(f"Duplicados Animal ID + fecha exacta: {len(dup_exact):,}")

# Solo eliminamos los duplicados de Animal ID + fecha exacta
merged = merged.unique(subset=["Animal ID", "DateTime"], keep="first")
print(f"Shape tras eliminar duplicados reales: {merged.shape}")

Filas duplicadas exactas: 0
Animal IDs duplicados: 9,914
Duplicados Animal ID + fecha exacta: 0
Shape tras eliminar duplicados reales: (111055, 31)
